In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [3]:
df_provs = pd.read_excel("_Provincias_Info.xlsx", sheet_name="df_LatLon")
df_provs["lat"] = df_provs["Lat"].astype(float)
df_provs["lon"] = df_provs["Lon"].astype(float)
df_provs["prov"] = df_provs["Nombre"].astype(str)
df_provs = df_provs[["prov", "lat", "lon"]]
df_provs

,prov,lat,lon
0,Madrid,40.495,-3.7170
1,Barcelona,41.730,1.9837
2,Valencia,39.370,-0.8000
3,Alicante,38.478,-0.5680
4,Sevilla,37.435,-5.6820
5,Málaga,36.813,-4.7250
6,Murcia,38.002,-1.4850
7,Cádiz,36.553,-5.7600
8,Islas Baleares,39.574,2.9124
9,Las Palmas,28.365,-14.5400


In [7]:
num_init = 1
num_end = 10


# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
city_names = df_provs.iloc[num_init:num_end+1]["prov"].to_list()
params = {
	"latitude":df_provs.iloc[num_init:num_end+1]["lat"].to_list(),
	"longitude":df_provs.iloc[num_init:num_end+1]["lon"].to_list(),
	"start_date": "2020-01-01",
	"end_date": "2026-03-24",
	"daily": ["weather_code", "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "apparent_temperature_mean", "wind_speed_10m_max", "wind_gusts_10m_max", "shortwave_radiation_sum", "sunrise", "sunset", "daylight_duration", "precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours", "surface_pressure_mean", "et0_fao_evapotranspiration_sum"],
}
responses = openmeteo.weather_api(url, params = params)

city_frames = []

# Process all locations and collect a dataframe per city
for city_name, response in zip(city_names, responses):
	print(f"\nCity: {city_name}")
	print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process daily data. The order of variables needs to be the same as requested.
	daily = response.Daily()
	daily_weather_code = daily.Variables(0).ValuesAsNumpy()
	daily_temperature_2m_mean = daily.Variables(1).ValuesAsNumpy()
	daily_temperature_2m_max = daily.Variables(2).ValuesAsNumpy()
	daily_temperature_2m_min = daily.Variables(3).ValuesAsNumpy()
	daily_apparent_temperature_mean = daily.Variables(4).ValuesAsNumpy()
	daily_wind_speed_10m_max = daily.Variables(5).ValuesAsNumpy()
	daily_wind_gusts_10m_max = daily.Variables(6).ValuesAsNumpy()
	daily_shortwave_radiation_sum = daily.Variables(7).ValuesAsNumpy()
	daily_sunrise = daily.Variables(8).ValuesInt64AsNumpy()
	daily_sunset = daily.Variables(9).ValuesInt64AsNumpy()
	daily_daylight_duration = daily.Variables(10).ValuesAsNumpy()
	daily_precipitation_sum = daily.Variables(11).ValuesAsNumpy()
	daily_rain_sum = daily.Variables(12).ValuesAsNumpy()
	daily_snowfall_sum = daily.Variables(13).ValuesAsNumpy()
	daily_precipitation_hours = daily.Variables(14).ValuesAsNumpy()
	daily_surface_pressure_mean = daily.Variables(15).ValuesAsNumpy()
	daily_et0_fao_evapotranspiration_sum = daily.Variables(16).ValuesAsNumpy()
	
	daily_data = {"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)}
	
	daily_data["weather_code"] = daily_weather_code
	daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
	daily_data["temperature_2m_max"] = daily_temperature_2m_max
	daily_data["temperature_2m_min"] = daily_temperature_2m_min
	daily_data["apparent_temperature_mean"] = daily_apparent_temperature_mean
	daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
	daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
	daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum
	daily_data["sunrise"] = daily_sunrise
	daily_data["sunset"] = daily_sunset
	daily_data["daylight_duration"] = daily_daylight_duration
	daily_data["precipitation_sum"] = daily_precipitation_sum
	daily_data["rain_sum"] = daily_rain_sum
	daily_data["snowfall_sum"] = daily_snowfall_sum
	daily_data["precipitation_hours"] = daily_precipitation_hours
	daily_data["surface_pressure_mean"] = daily_surface_pressure_mean
	daily_data["et0_fao_evapotranspiration_sum"] = daily_et0_fao_evapotranspiration_sum
	
	daily_city_df = pd.DataFrame(data = daily_data)
	daily_city_df["city"] = city_name
	city_frames.append(daily_city_df)

daily_dataframe_2 = pd.concat(city_frames, ignore_index = True)
daily_dataframe_2

OpenMeteoRequestsError: failed to request 'https://archive-api.open-meteo.com/v1/archive': {'reason': 'Daily API request limit exceeded. Please try again tomorrow.', 'error': True}

In [35]:
extract_date = pd.to_datetime("today")
place = "top11-51_cities"
dates_from_to = "2020-2026"
csv_title = f"daily_{place}_{dates_from_to}_{extract_date.strftime('%Y-%m-%d')}_extracted.csv"

daily_dataframe_2.to_csv(csv_title, index = False)
daily_dataframe_2

,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,wind_speed_10m_max,wind_gusts_10m_max,shortwave_radiation_sum,sunrise,sunset,daylight_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,surface_pressure_mean,et0_fao_evapotranspiration_sum,city
0,2020-01-01 00:00:00+00:00,3.0,5.487501,10.250000,1.750,3.083325,16.520823,29.519999,7.890000,1577865925,1577898520,32593.833984,0.000000,0.000000,0.0,0.0,987.083923,0.845411,La Coruña
1,2020-01-02 00:00:00+00:00,3.0,7.464584,8.800000,5.100,2.961903,31.259941,63.720001,3.240000,1577952329,1577984973,32642.673828,0.000000,0.000000,0.0,0.0,984.536316,0.381227,La Coruña
2,2020-01-03 00:00:00+00:00,51.0,9.210417,11.700000,6.400,6.982733,24.363251,46.079998,5.890000,1578038731,1578071428,32695.638672,1.500000,1.500000,0.0,7.0,989.133301,0.724495,La Coruña
3,2020-01-04 00:00:00+00:00,3.0,5.833333,10.050000,2.750,3.360067,11.183201,24.840000,7.420000,1578125131,1578157885,32752.623047,0.000000,0.000000,0.0,0.0,991.555664,0.798509,La Coruña
4,2020-01-05 00:00:00+00:00,2.0,5.474999,10.500000,2.100,2.847437,13.755580,25.919998,7.970000,1578211529,1578244344,32813.523438,0.000000,0.000000,0.0,0.0,986.742004,0.850397,La Coruña
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93270,2026-03-20 00:00:00+00:00,61.0,14.105999,15.481000,13.531,10.870917,38.312946,74.879997,6.010000,1773987913,1774031597,43681.804688,14.700002,14.700002,0.0,21.0,1004.321777,1.094613,Ceuta
93271,2026-03-21 00:00:00+00:00,51.0,14.510169,15.781000,13.831,11.446494,27.504095,56.519997,9.810000,1774074228,1774118048,43818.050781,1.200000,1.200000,0.0,10.0,1004.847229,1.689053,Ceuta
93272,2026-03-22 00:00:00+00:00,51.0,14.643500,16.881001,13.331,12.026841,24.948025,50.399998,16.730000,1774160543,1774204499,43953.625000,0.100000,0.100000,0.0,1.0,1005.633118,2.503936,Ceuta
93273,2026-03-23 00:00:00+00:00,1.0,14.889335,17.731001,13.031,13.187428,25.600531,49.680000,21.120001,1774246859,1774290949,44088.609375,0.000000,0.000000,0.0,0.0,1008.261047,2.969439,Ceuta
